# Topic 11: Dynamic Programming

**Goal**: Understand the DP mindset, master top-down and bottom-up approaches, solve the classic patterns.  
**Time**: ~10-12 hours  
**Prereqs**: Topics 2, 6

---

## Why Is DP Hard?

DP isn't hard because the code is complex — it's hard because recognizing the **PATTERN** is hard. Two signs a problem needs DP:

1. **Overlapping subproblems**: same computation happens multiple times
2. **Optimal substructure**: optimal solution uses optimal solutions to subproblems

```
              fib(5)
             /      \
         fib(4)      fib(3)        ← fib(3) computed TWICE
        /     \      /    \
     fib(3)  fib(2) fib(2) fib(1)  ← fib(2) computed THREE TIMES
     /   \
  fib(2) fib(1)

  Without DP: recalculate everything  →  EXPONENTIAL
  With DP:    store & reuse results   →  LINEAR
```

---

## The Two Approaches

```
TOP-DOWN (Memoization):              BOTTOM-UP (Tabulation):
  Start from the big problem           Start from smallest subproblems
  Break down recursively               Build up iteratively
  Cache results in a dict              Fill a table row by row
  Feels like "smart recursion"         Feels like "filling a spreadsheet"

     fib(5)                            dp = [0, 1, _, _, _, _]
       ↓ calls fib(4), fib(3)              [0, 1, 1, _, _, _]
       ↓ but fib(3) is cached!             [0, 1, 1, 2, _, _]
       ↓ returns instantly                 [0, 1, 1, 2, 3, _]
     = 5                                   [0, 1, 1, 2, 3, 5]  ✓

  BOTH give the same answer.
  BOTH are O(n) time, O(n) space.
  Pick whichever feels more natural for the problem.
```

---

## Part 1: Building Intuition — Fibonacci

Fibonacci is the gateway drug to DP. Every DP technique shows up here in its simplest form.

```
fib(0) = 0
fib(1) = 1
fib(n) = fib(n-1) + fib(n-2)   for n >= 2

Sequence: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ...
```

We'll solve this FOUR different ways, each one better than the last.

In [ ]:
# === APPROACH 1: Naive Recursion — O(2^n) ===

call_count = 0

def fib_naive(n):
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

print("=== Naive Recursive Fibonacci ===")
for n in [5, 10, 20, 30]:
    call_count = 0
    result = fib_naive(n)
    print(f"  fib({n:2d}) = {result:>10,}  |  function calls: {call_count:>15,}")

print()
print("  n=30 needs 2.6 MILLION calls.")
print("  n=50 would take MINUTES. n=100 would outlive the sun.")
print("  This is O(2^n) — completely unusable.")

In [ ]:
# === APPROACH 2: Top-Down with Memoization — O(n) ===

call_count = 0

def fib_memo(n, memo=None):
    global call_count
    call_count += 1
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

print("=== Memoized Fibonacci ===")
for n in [5, 10, 20, 30, 50, 100]:
    call_count = 0
    result = fib_memo(n)
    print(f"  fib({n:3d}) = {result:>25,}  |  calls: {call_count:>6}")

print()
print("  fib(100) — INSTANT. Only ~200 calls instead of heat-death-of-universe.")
print("  Same recursion, just remember what you've already computed.")

In [ ]:
# === APPROACH 3: Bottom-Up Table — O(n) time, O(n) space ===

def fib_table(n):
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

print("=== Bottom-Up Table Fibonacci ===")
print("  Watching the dp array fill up:\n")

n = 10
dp = [0] * (n + 1)
dp[1] = 1
print(f"  Base cases: dp = {dp}")
for i in range(2, n + 1):
    dp[i] = dp[i - 1] + dp[i - 2]
    print(f"  dp[{i:2d}] = dp[{i-1}] + dp[{i-2}] = {dp[i-1]} + {dp[i-2]} = {dp[i]}")
    print(f"         dp = {dp}")

print(f"\n  Answer: fib({n}) = {dp[n]}")
print("  No recursion. Just fill the table left to right.")

In [ ]:
# === APPROACH 4: Space-Optimized — O(n) time, O(1) space ===

def fib_optimized(n):
    if n <= 1:
        return n
    prev2, prev1 = 0, 1
    for i in range(2, n + 1):
        curr = prev1 + prev2
        prev2, prev1 = prev1, curr
    return prev1

print("=== Space-Optimized Fibonacci ===")
print("  We only ever need the previous TWO values.\n")

n = 10
prev2, prev1 = 0, 1
print(f"  Start: prev2=0, prev1=1")
for i in range(2, n + 1):
    curr = prev1 + prev2
    print(f"  i={i:2d}: curr = {prev1} + {prev2} = {curr}  →  prev2={prev1}, prev1={curr}")
    prev2, prev1 = prev1, curr

print(f"\n  Answer: fib({n}) = {prev1}")
print("  No array at all. Just two variables sliding forward.")

---

## The DP Workflow — Apply This to EVERY Problem

```
  STEP 1             STEP 2              STEP 3              STEP 4
  Brute Force    →   Memoization     →   Tabulation      →   Space Optimize
  (Recursion)        (Top-Down)          (Bottom-Up)         (Reduce memory)

  O(2^n)             O(n)                O(n)                O(n) time
  Exponential        O(n) space          O(n) space          O(1) space ← !!

  ┌──────────┐    ┌──────────┐       ┌──────────┐       ┌──────────┐
  │ Recursion │ →  │ + Cache  │   →   │ For loop │   →   │ Trim     │
  │ tree      │    │ (dict)   │       │ + table  │       │ to only  │
  │ explodes  │    │ prunes   │       │ no stack │       │ what you │
  │           │    │ branches │       │ overflow │       │ need     │
  └──────────┘    └──────────┘       └──────────┘       └──────────┘
```

**Not every problem can be space-optimized**, but the first three steps always apply.

---

## Part 2: The DP Framework

For **every** DP problem, answer these four questions:

```
┌─────────────────────────────────────────────────────────────────┐
│                    THE DP FRAMEWORK                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. STATE:      What info describes a subproblem?               │
│                 What does dp[i] (or dp[i][j]) represent?        │
│                                                                 │
│  2. RECURRENCE: How does dp[i] relate to smaller subproblems?   │
│                 This is the heart of the solution.              │
│                                                                 │
│  3. BASE CASE:  What's the smallest subproblem I can solve      │
│                 directly, without the recurrence?               │
│                                                                 │
│  4. ANSWER:     Which cell in the table is my final answer?     │
│                 dp[n]? dp[n][m]? max(dp)?                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

**Example with Fibonacci:**

| Question | Answer |
|----------|--------|
| State | `dp[i]` = the i-th Fibonacci number |
| Recurrence | `dp[i] = dp[i-1] + dp[i-2]` |
| Base case | `dp[0] = 0`, `dp[1] = 1` |
| Answer | `dp[n]` |

Fill this out BEFORE writing code. Every time.

---

## Problem 1: Climbing Stairs (LC #70)

You're climbing a staircase with `n` steps. Each time you can take **1 or 2 steps**. How many distinct ways can you reach the top?

```
                    ┌───┐
                 ┌──┤ 4 │  ← GOAL
              ┌──┤  └───┘
           ┌──┤  │   3
        ┌──┤  │  └──────
     ┌──┤  │  │   2
  ┌──┤  │  └──┘
  │  │  │   1
  │  └──┘                  From step i, you came from step i-1 OR step i-2.
  │   0  ← START          So: dp[i] = dp[i-1] + dp[i-2]
  └──────
                           This IS Fibonacci!
```

| Question | Answer |
|----------|--------|
| State | `dp[i]` = number of ways to reach step `i` |
| Recurrence | `dp[i] = dp[i-1] + dp[i-2]` |
| Base case | `dp[0] = 1` (one way to stand still), `dp[1] = 1` |
| Answer | `dp[n]` |

In [ ]:
def climb_stairs(n):
    if n <= 1:
        return 1
    dp = [0] * (n + 1)
    dp[0], dp[1] = 1, 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

print("=== Climbing Stairs ===")
for n in [2, 3, 4, 5, 6, 10]:
    dp = [0] * (n + 1)
    dp[0], dp[1] = 1, 1
    steps = [f"dp[0]=1, dp[1]=1"]
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    print(f"  n={n:2d}: dp = {dp[:n+1]}  →  {dp[n]} ways")

print("\n  Trace for n=5:")
dp = [0] * 6
dp[0], dp[1] = 1, 1
print(f"  Base: dp = {dp}")
for i in range(2, 6):
    dp[i] = dp[i - 1] + dp[i - 2]
    print(f"  dp[{i}] = dp[{i-1}]({dp[i-1]}) + dp[{i-2}]({dp[i-2] if i >= 3 else dp[i-2]}) = {dp[i]}  →  dp = {dp}")
print(f"\n  Answer: {dp[5]} distinct ways to climb 5 stairs")

---

## Problem 2: Coin Change (LC #322)

Given coins of certain denominations, find the **minimum** number of coins to make a target amount. Return -1 if impossible.

```
coins = [1, 3, 4], amount = 6

dp[i] = minimum coins to make amount i

amt:  0    1    2    3    4    5    6
     ┌────┬────┬────┬────┬────┬────┬────┐
dp:  │  0 │  1 │  2 │  1 │  1 │  2 │  2 │
     └────┴────┴────┴────┴────┴────┴────┘
       ↑    ↑    ↑    ↑    ↑    ↑    ↑
       0    1   1+1   3    4   1+4  3+3
      base  (1) (1,1) (3) (4) (1,4)(3,3)

For each amount, try every coin:
  dp[6] = min(dp[6-1]+1, dp[6-3]+1, dp[6-4]+1)
        = min(dp[5]+1,   dp[3]+1,   dp[2]+1)
        = min(  3,         2,          3   )
        = 2  (use coin 3 twice: 3+3)
```

| Question | Answer |
|----------|--------|
| State | `dp[i]` = min coins to make amount `i` |
| Recurrence | `dp[i] = min(dp[i - coin] + 1)` for each coin |
| Base case | `dp[0] = 0` (zero coins to make zero) |
| Answer | `dp[amount]` (or -1 if still infinity) |

In [ ]:
def coin_change(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1
    return dp[amount] if dp[amount] != float('inf') else -1

print("=== Coin Change (Bottom-Up) ===")
coins = [1, 3, 4]
amount = 6
print(f"  Coins: {coins}, Amount: {amount}\n")

dp = [float('inf')] * (amount + 1)
dp[0] = 0
print(f"  Base: dp = {[x if x != float('inf') else '∞' for x in dp]}")

for i in range(1, amount + 1):
    best_coin = None
    for coin in coins:
        if coin <= i and dp[i - coin] + 1 < dp[i]:
            dp[i] = dp[i - coin] + 1
            best_coin = coin
    display = [x if x != float('inf') else '∞' for x in dp]
    coin_str = f"used coin {best_coin}" if best_coin else "unreachable"
    print(f"  dp[{i}] = {dp[i] if dp[i] != float('inf') else '∞':>3}  ({coin_str:16s})  dp = {display}")

print(f"\n  Answer: {dp[amount]} coins")

print("\n--- More test cases ---")
for coins, amount in [([1, 5, 10, 25], 30), ([2], 3), ([1, 2, 5], 11)]:
    result = coin_change(coins, amount)
    print(f"  coins={str(coins):15s} amount={amount:3d}  →  {result} coins")

In [ ]:
def coin_change_memo(coins, amount):
    memo = {}
    def dp(remaining):
        if remaining == 0:
            return 0
        if remaining < 0:
            return float('inf')
        if remaining in memo:
            return memo[remaining]
        best = float('inf')
        for coin in coins:
            best = min(best, dp(remaining - coin) + 1)
        memo[remaining] = best
        return best

    result = dp(amount)
    return result if result != float('inf') else -1, memo

print("=== Coin Change (Top-Down / Memoization) ===")
coins = [1, 3, 4]
amount = 6
result, memo = coin_change_memo(coins, amount)
print(f"  Coins: {coins}, Amount: {amount}")
print(f"  Result: {result} coins\n")
print(f"  Memo cache (what was stored):")
for k in sorted(memo.keys()):
    v = memo[k] if memo[k] != float('inf') else '∞'
    print(f"    memo[{k}] = {v}")
print("\n  Same answer, different traversal order.")
print("  Top-down: only computes what's needed (lazy).")
print("  Bottom-up: computes everything from 0..amount (eager).")

---

## Problem 3: 0/1 Knapsack

THE classic DP problem. Given items with weights and values, maximize total value within a weight limit. Each item can be taken **at most once**.

```
Items: [(wt=1, val=1), (wt=3, val=4), (wt=4, val=5), (wt=5, val=7)]
Capacity = 7

dp[i][w] = max value using items 0..i with capacity w

For each item, two choices:
  SKIP it:  dp[i][w] = dp[i-1][w]
  TAKE it:  dp[i][w] = dp[i-1][w - wt[i]] + val[i]   (if wt[i] <= w)

        capacity →
          0  1  2  3  4  5  6  7
       0 [0, 0, 0, 0, 0, 0, 0, 0]  ← no items
item 1 [0, 1, 1, 1, 1, 1, 1, 1]  ← (wt=1, val=1)
item 2 [0, 1, 1, 4, 5, 5, 5, 5]  ← (wt=3, val=4)
item 3 [0, 1, 1, 4, 5, 6, 6, 9]  ← (wt=4, val=5)
item 4 [0, 1, 1, 4, 5, 7, 8, 9]  ← (wt=5, val=7)
                               ↑
                          Answer = 9

Best: items 2+3 (wt=3+4=7, val=4+5=9)
```

| Question | Answer |
|----------|--------|
| State | `dp[i][w]` = max value with items 0..i, capacity w |
| Recurrence | `dp[i][w] = max(dp[i-1][w], dp[i-1][w-wt[i]] + val[i])` |
| Base case | `dp[0][w] = 0` for all w (no items = no value) |
| Answer | `dp[n][capacity]` |

In [ ]:
def knapsack_2d(weights, values, capacity):
    n = len(weights)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]
            if weights[i - 1] <= w:
                dp[i][w] = max(dp[i][w], dp[i - 1][w - weights[i - 1]] + values[i - 1])
    return dp[n][capacity], dp

weights = [1, 3, 4, 5]
values  = [1, 4, 5, 7]
capacity = 7

print("=== 0/1 Knapsack (2D Table) ===")
print(f"  Items: {list(zip(weights, values))} (weight, value)")
print(f"  Capacity: {capacity}\n")

result, dp = knapsack_2d(weights, values, capacity)

header = "     " + "".join(f"{w:>4}" for w in range(capacity + 1))
print(f"  {header}  ← capacity")
labels = ["  none"] + [f"  item{i}" for i in range(1, len(weights) + 1)]
for i, row in enumerate(dp):
    item_info = f"(wt={weights[i-1]}, val={values[i-1]})" if i > 0 else "(no items)"
    print(f"{labels[i]} {row}  ← {item_info}")

print(f"\n  Max value: {result}")

# Backtrack to find which items were selected
i, w = len(weights), capacity
selected = []
while i > 0 and w > 0:
    if dp[i][w] != dp[i - 1][w]:
        selected.append(i)
        w -= weights[i - 1]
    i -= 1

print(f"  Selected items: {selected[::-1]}")
print(f"  Total weight: {sum(weights[i-1] for i in selected)}")
print(f"  Total value:  {sum(values[i-1] for i in selected)}")

In [ ]:
def knapsack_1d(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(weights)):
        for w in range(capacity, weights[i] - 1, -1):
            dp[w] = max(dp[w], dp[w - weights[i]] + values[i])
    return dp[capacity]

print("=== 0/1 Knapsack (1D Space-Optimized) ===")
print("  Key insight: each row only depends on the PREVIOUS row.")
print("  Iterate capacity RIGHT TO LEFT to avoid using updated values.\n")

weights = [1, 3, 4, 5]
values  = [1, 4, 5, 7]
capacity = 7

dp = [0] * (capacity + 1)
print(f"  Start:  dp = {dp}")
for i in range(len(weights)):
    for w in range(capacity, weights[i] - 1, -1):
        dp[w] = max(dp[w], dp[w - weights[i]] + values[i])
    print(f"  Item {i+1} (wt={weights[i]}, val={values[i]}):  dp = {dp}")

print(f"\n  Answer: {dp[capacity]}")
print(f"  Same result, O(capacity) space instead of O(n × capacity).")

---

## Problem 4: Longest Common Subsequence (LC #1143)

Given two strings, find the length of their longest common subsequence. A subsequence doesn't need to be contiguous.

```
text1 = "abcde",  text2 = "ace"
LCS = "ace"  →  length 3

dp[i][j] = LCS of text1[0..i-1] and text2[0..j-1]

If chars match:    dp[i][j] = dp[i-1][j-1] + 1
If chars differ:   dp[i][j] = max(dp[i-1][j], dp[i][j-1])

         ""  a   c   e
    ""  [ 0, 0,  0,  0 ]
     a  [ 0, 1,  1,  1 ]   a==a → dp[0][0]+1 = 1
     b  [ 0, 1,  1,  1 ]   b≠a,c,e → carry forward
     c  [ 0, 1,  2,  2 ]   c==c → dp[1][1]+1 = 2
     d  [ 0, 1,  2,  2 ]   d≠a,c,e → carry forward
     e  [ 0, 1,  2,  3 ]   e==e → dp[3][2]+1 = 3  ✓
                        ↑
                   Answer = 3
```

| Question | Answer |
|----------|--------|
| State | `dp[i][j]` = LCS length of `text1[0..i-1]` and `text2[0..j-1]` |
| Recurrence | match: `dp[i-1][j-1]+1`, else: `max(dp[i-1][j], dp[i][j-1])` |
| Base case | `dp[0][j] = dp[i][0] = 0` (empty string has LCS 0) |
| Answer | `dp[m][n]` |

In [ ]:
def lcs(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n], dp

print("=== Longest Common Subsequence ===")
text1, text2 = "abcde", "ace"
result, dp = lcs(text1, text2)

print(f"  text1 = \"{text1}\",  text2 = \"{text2}\"\n")
header = "       \"\"  " + "  ".join(f"{c:>2}" for c in text2)
print(f"  {header}")
labels = ['""'] + list(text1)
for i, row in enumerate(dp):
    label = f"{labels[i]:>4}"
    print(f"  {label}  {row}")

print(f"\n  LCS length: {result}")

print("\n--- Trace of key decisions ---")
for i in range(1, len(text1) + 1):
    for j in range(1, len(text2) + 1):
        if text1[i-1] == text2[j-1]:
            print(f"  dp[{i}][{j}]: '{text1[i-1]}' == '{text2[j-1]}'  →  dp[{i-1}][{j-1}]+1 = {dp[i][j]}")
        else:
            print(f"  dp[{i}][{j}]: '{text1[i-1]}' ≠  '{text2[j-1]}'  →  max(dp[{i-1}][{j}], dp[{i}][{j-1}]) = max({dp[i-1][j]}, {dp[i][j-1]}) = {dp[i][j]}")

In [ ]:
def lcs_with_reconstruct(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i - 1] == text2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Backtrack to find the actual subsequence
    result = []
    i, j = m, n
    while i > 0 and j > 0:
        if text1[i - 1] == text2[j - 1]:
            result.append(text1[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] > dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(result))

print("=== LCS Reconstruction ===")
pairs = [("abcde", "ace"), ("abc", "abc"), ("abc", "def"), ("AGGTAB", "GXTXAYB")]
for t1, t2 in pairs:
    subseq = lcs_with_reconstruct(t1, t2)
    print(f"  \"{t1}\" vs \"{t2}\"  →  LCS = \"{subseq}\" (length {len(subseq)})")

print("\n--- Backtracking walkthrough for 'AGGTAB' vs 'GXTXAYB' ---")
t1, t2 = "AGGTAB", "GXTXAYB"
m, n = len(t1), len(t2)
dp = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if t1[i-1] == t2[j-1]:
            dp[i][j] = dp[i-1][j-1] + 1
        else:
            dp[i][j] = max(dp[i-1][j], dp[i][j-1])

i, j = m, n
path = []
while i > 0 and j > 0:
    if t1[i-1] == t2[j-1]:
        path.append(f"  ({i},{j}): '{t1[i-1]}' == '{t2[j-1]}'  →  take it, move diagonal")
        i -= 1
        j -= 1
    elif dp[i-1][j] > dp[i][j-1]:
        path.append(f"  ({i},{j}): '{t1[i-1]}' ≠  '{t2[j-1]}'  →  move up (dp[{i-1}][{j}]={dp[i-1][j]} > dp[{i}][{j-1}]={dp[i][j-1]})")
        i -= 1
    else:
        path.append(f"  ({i},{j}): '{t1[i-1]}' ≠  '{t2[j-1]}'  →  move left")
        j -= 1
for step in path:
    print(step)

---

## Problem 5: Longest Increasing Subsequence (LC #300)

Given an array, find the length of the longest strictly increasing subsequence.

```
nums = [10, 9, 2, 5, 3, 7, 101, 18]

dp[i] = length of LIS ending at index i

For each i, look at ALL j < i where nums[j] < nums[i]:
  dp[i] = max(dp[j] + 1) for all valid j

idx:    0    1    2    3    4    5    6     7
nums: [10,   9,   2,   5,   3,   7, 101,  18]
dp:   [ 1,   1,   1,   2,   2,   3,   4,   4]
                       ↑         ↑         ↑
                     2→5       2→5→7    2→5→7→18
                    or 2→3    or 2→3→7  or 2→3→7→18

Answer: max(dp) = 4  →  [2, 3, 7, 101] or [2, 3, 7, 18] etc.
```

| Question | Answer |
|----------|--------|
| State | `dp[i]` = length of LIS ending at index `i` |
| Recurrence | `dp[i] = max(dp[j] + 1)` for all `j < i` where `nums[j] < nums[i]` |
| Base case | `dp[i] = 1` for all `i` (each element alone is a subsequence) |
| Answer | `max(dp)` |

In [ ]:
def length_of_lis(nums):
    n = len(nums)
    dp = [1] * n
    for i in range(1, n):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

print("=== Longest Increasing Subsequence — O(n²) ===")
nums = [10, 9, 2, 5, 3, 7, 101, 18]
print(f"  nums = {nums}\n")

n = len(nums)
dp = [1] * n
for i in range(1, n):
    candidates = []
    for j in range(i):
        if nums[j] < nums[i]:
            candidates.append((j, nums[j], dp[j]))
            dp[i] = max(dp[i], dp[j] + 1)
    cand_str = ", ".join(f"j={j}(val={v},dp={d})" for j, v, d in candidates) if candidates else "none"
    print(f"  i={i} nums[{i}]={nums[i]:>3}: smaller predecessors: [{cand_str}] → dp[{i}]={dp[i]}")

print(f"\n  dp   = {dp}")
print(f"  nums = {nums}")
print(f"  LIS length: {max(dp)}")

print("\n--- More tests ---")
for arr in [[0, 1, 0, 3, 2, 3], [7, 7, 7, 7], [1, 3, 6, 7, 9, 4, 10, 5, 6]]:
    print(f"  {str(arr):40s} → LIS = {length_of_lis(arr)}")

In [ ]:
import bisect

def lis_binary_search(nums):
    tails = []
    for x in nums:
        pos = bisect.bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
    return len(tails)

print("=== LIS with Binary Search — O(n log n) ===")
print("  Maintain array 'tails' where tails[i] = smallest tail of any")
print("  increasing subsequence of length i+1.\n")

nums = [10, 9, 2, 5, 3, 7, 101, 18]
print(f"  nums = {nums}\n")

tails = []
for x in nums:
    pos = bisect.bisect_left(tails, x)
    action = "append" if pos == len(tails) else f"replace tails[{pos}]"
    if pos == len(tails):
        tails.append(x)
    else:
        tails[pos] = x
    print(f"  x={x:>3}: bisect_left={pos}, {action:20s} → tails = {tails}")

print(f"\n  LIS length: {len(tails)}")
print("  NOTE: 'tails' is NOT the actual LIS — it's a helper structure.")
print("  But its LENGTH equals the LIS length.")

---

## Problem 6: Unique Paths (LC #62)

An `m × n` grid. Start at top-left, reach bottom-right. Can only move **right** or **down**. How many unique paths?

```
m=3, n=4

dp[i][j] = number of paths to reach cell (i, j)

    0    1    2    3
 0 [ 1,   1,   1,   1 ]   ← top row: only one way (go right)
 1 [ 1,   2,   3,   4 ]   
 2 [ 1,   3,   6,  10 ]   ← bottom-right = 10 paths
                    ↑
           dp[i][j] = dp[i-1][j] + dp[i][j-1]
                      (from above) + (from left)

Example path: →→→↓↓  or  ↓→→↓→  etc.
```

| Question | Answer |
|----------|--------|
| State | `dp[i][j]` = paths to reach cell `(i, j)` |
| Recurrence | `dp[i][j] = dp[i-1][j] + dp[i][j-1]` |
| Base case | First row and first column are all 1 |
| Answer | `dp[m-1][n-1]` |

In [ ]:
def unique_paths(m, n):
    dp = [[1] * n for _ in range(m)]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp[m - 1][n - 1], dp

print("=== Unique Paths ===")
m, n = 3, 4
result, dp = unique_paths(m, n)
print(f"  Grid: {m} × {n}\n")

for i, row in enumerate(dp):
    cells = "".join(f"{v:>5}" for v in row)
    print(f"  row {i}: [{cells} ]")

print(f"\n  Answer: {result} unique paths")

print("\n--- Trace for key cells ---")
for i in range(1, m):
    for j in range(1, n):
        print(f"  dp[{i}][{j}] = dp[{i-1}][{j}]({dp[i-1][j]}) + dp[{i}][{j-1}]({dp[i][j-1]}) = {dp[i][j]}")

print("\n--- More grids ---")
for m, n in [(1, 1), (2, 2), (3, 3), (3, 7), (7, 3)]:
    r, _ = unique_paths(m, n)
    print(f"  {m}×{n}: {r} paths")

---

## Problem 7: Minimum Path Sum (LC #64)

Same grid, but each cell has a cost. Find the path from top-left to bottom-right with **minimum total cost**.

```
grid:              dp (min cost to reach each cell):
  [1, 3, 1]         [ 1,  4,  5]
  [1, 5, 1]    →    [ 2,  7,  6]
  [4, 2, 1]         [ 6,  8,  7]  ← answer = 7

  Optimal path: 1→3→1→1→1 = 7  (right, right, down, down)
  Wait — 1→1→5→1→1 = 9. Nope, greedy doesn't work here.
  Actual best: 1→3→1→1→1 = 7 (→→↓↓) or 1→1→4→2→1 = nah.
  DP gives us the globally optimal answer.
```

| Question | Answer |
|----------|--------|
| State | `dp[i][j]` = min cost to reach cell `(i, j)` |
| Recurrence | `dp[i][j] = grid[i][j] + min(dp[i-1][j], dp[i][j-1])` |
| Base case | `dp[0][0] = grid[0][0]`, first row/col are cumulative sums |
| Answer | `dp[m-1][n-1]` |

In [ ]:
def min_path_sum(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    dp[0][0] = grid[0][0]
    for j in range(1, n):
        dp[0][j] = dp[0][j - 1] + grid[0][j]
    for i in range(1, m):
        dp[i][0] = dp[i - 1][0] + grid[i][0]
    for i in range(1, m):
        for j in range(1, n):
            dp[i][j] = grid[i][j] + min(dp[i - 1][j], dp[i][j - 1])
    return dp[m - 1][n - 1], dp

print("=== Minimum Path Sum ===")
grid = [
    [1, 3, 1],
    [1, 5, 1],
    [4, 2, 1]
]

print("  Grid:")
for row in grid:
    print(f"    {row}")

result, dp = min_path_sum(grid)
print("\n  DP table (min cost to reach each cell):")
for row in dp:
    print(f"    {row}")

print(f"\n  Min path sum: {result}")

print("\n--- Trace ---")
m, n = len(grid), len(grid[0])
print(f"  dp[0][0] = {grid[0][0]} (start)")
for j in range(1, n):
    print(f"  dp[0][{j}] = dp[0][{j-1}]({dp[0][j-1]}) + grid[0][{j}]({grid[0][j]}) = {dp[0][j]}")
for i in range(1, m):
    print(f"  dp[{i}][0] = dp[{i-1}][0]({dp[i-1][0]}) + grid[{i}][0]({grid[i][0]}) = {dp[i][0]}")
for i in range(1, m):
    for j in range(1, n):
        from_above, from_left = dp[i-1][j], dp[i][j-1]
        chosen = "above" if from_above <= from_left else "left"
        print(f"  dp[{i}][{j}] = grid({grid[i][j]}) + min(above={from_above}, left={from_left}) = {dp[i][j]}  ← from {chosen}")

---

## Problem 8: House Robber (LC #198)

Rob houses along a street. Can't rob two adjacent houses. Maximize total money.

```
nums = [2, 7, 9, 3, 1]

At each house, two choices:
  SKIP this house:  dp[i] = dp[i-1]
  ROB this house:   dp[i] = dp[i-2] + nums[i]

dp[i] = max(dp[i-1], dp[i-2] + nums[i])

idx:    0    1    2    3    4
nums: [ 2,   7,   9,   3,   1]
dp:   [ 2,   7,  11,  11,  12]
        ↑    ↑    ↑         ↑
       rob  rob  rob+      rob+
             7   2+9=11   11+1=12

Answer: 12  (rob houses 0, 2, 4 → 2+9+1=12)
```

| Question | Answer |
|----------|--------|
| State | `dp[i]` = max money robbing from houses `0..i` |
| Recurrence | `dp[i] = max(dp[i-1], dp[i-2] + nums[i])` |
| Base case | `dp[0] = nums[0]`, `dp[1] = max(nums[0], nums[1])` |
| Answer | `dp[n-1]` |

In [ ]:
def house_robber(nums):
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    dp[1] = max(nums[0], nums[1])
    for i in range(2, n):
        dp[i] = max(dp[i - 1], dp[i - 2] + nums[i])
    return dp[n - 1], dp

print("=== House Robber ===")
nums = [2, 7, 9, 3, 1]
result, dp = house_robber(nums)
print(f"  Houses: {nums}\n")

print(f"  dp[0] = {nums[0]} (only one house, rob it)")
print(f"  dp[1] = max({nums[0]}, {nums[1]}) = {dp[1]} (pick the richer house)")
for i in range(2, len(nums)):
    skip = dp[i - 1]
    rob = dp[i - 2] + nums[i]
    choice = "ROB" if rob > skip else "SKIP"
    print(f"  dp[{i}] = max(skip={skip}, rob=dp[{i-2}]+{nums[i]}={rob}) = {dp[i]}  → {choice}")

print(f"\n  dp   = {dp}")
print(f"  nums = {nums}")
print(f"  Max loot: {result}")

print("\n--- Space-optimized (O(1)) ---")
def house_robber_opt(nums):
    prev2, prev1 = 0, 0
    for x in nums:
        curr = max(prev1, prev2 + x)
        prev2, prev1 = prev1, curr
    return prev1

for arr in [[2, 7, 9, 3, 1], [1, 2, 3, 1], [2, 1, 1, 2]]:
    print(f"  {str(arr):20s} → {house_robber_opt(arr)}")

---

## Problem 9: Edit Distance (LC #72)

Minimum operations to convert `word1` into `word2`. Allowed: **insert**, **delete**, **replace** a character.

```
word1 = "horse",  word2 = "ros"

dp[i][j] = min edits to convert word1[0..i-1] to word2[0..j-1]

If chars match:  dp[i][j] = dp[i-1][j-1]        (no edit needed)
If chars differ: dp[i][j] = 1 + min(
                   dp[i-1][j],     ← delete from word1
                   dp[i][j-1],     ← insert into word1
                   dp[i-1][j-1]    ← replace
                 )

         ""  r   o   s
    ""  [ 0, 1,  2,  3 ]
     h  [ 1, 1,  2,  3 ]
     o  [ 2, 2,  1,  2 ]
     r  [ 3, 2,  2,  2 ]
     s  [ 4, 3,  3,  2 ]
     e  [ 5, 4,  4,  3 ]  ← answer = 3

Operations: horse → rorse (replace h→r) → rose (delete r) → ros (delete e)
```

| Question | Answer |
|----------|--------|
| State | `dp[i][j]` = min edits for `word1[0..i-1]` → `word2[0..j-1]` |
| Recurrence | match: `dp[i-1][j-1]`, else `1 + min(del, ins, rep)` |
| Base case | `dp[i][0] = i` (delete all), `dp[0][j] = j` (insert all) |
| Answer | `dp[m][n]` |

In [ ]:
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n], dp

print("=== Edit Distance ===")
word1, word2 = "horse", "ros"
result, dp = edit_distance(word1, word2)
print(f"  \"{word1}\" → \"{word2}\"\n")

header = "       \"\"  " + "  ".join(f"{c:>2}" for c in word2)
print(f"  {header}")
labels = ['""'] + list(word1)
for i, row in enumerate(dp):
    print(f"  {labels[i]:>4}  {row}")

print(f"\n  Edit distance: {result}")

print("\n--- Key decisions trace ---")
m, n = len(word1), len(word2)
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if word1[i-1] == word2[j-1]:
            print(f"  dp[{i}][{j}]: '{word1[i-1]}'=='{word2[j-1]}'  →  dp[{i-1}][{j-1}] = {dp[i][j]}  (free)")
        else:
            d, ins, rep = dp[i-1][j], dp[i][j-1], dp[i-1][j-1]
            ops = {d: 'delete', ins: 'insert', rep: 'replace'}
            best = min(d, ins, rep)
            print(f"  dp[{i}][{j}]: '{word1[i-1]}'≠'{word2[j-1]}'  →  1+min(del={d},ins={ins},rep={rep}) = {dp[i][j]}  ({ops[best]})")

print("\n--- More tests ---")
for w1, w2 in [("intention", "execution"), ("abc", "abc"), ("", "abc"), ("kitten", "sitting")]:
    r, _ = edit_distance(w1, w2)
    print(f"  \"{w1}\" → \"{w2}\":  {r} edits")

---

## Problem 10: Best Time to Buy and Sell Stock with Cooldown (LC #309)

You can buy/sell stock multiple times, but after selling you must wait one day (cooldown) before buying again.

```
This is STATE MACHINE DP. Three states:

  ┌─────────┐     sell      ┌──────────┐    cooldown   ┌──────────────┐
  │  HOLD   │ ─────────→  │   SOLD    │ ──────────→ │   COOLDOWN   │
  │ (have   │              │ (just     │              │ (waiting,    │
  │  stock) │              │  sold)    │              │  can't buy)  │
  └─────────┘              └──────────┘              └──────────────┘
       ↑                                                    │
       │                      buy                           │
       └────────────────────────────────────────────────────┘
       │
       └──── hold (do nothing, keep stock) ────→ self loop

  hold[i]     = max(hold[i-1],     cooldown[i-1] - prices[i])  ← keep or buy
  sold[i]     = hold[i-1] + prices[i]                          ← sell
  cooldown[i] = max(cooldown[i-1], sold[i-1])                  ← wait or transition
```

| Question | Answer |
|----------|--------|
| State | `hold[i]`, `sold[i]`, `cooldown[i]` — profit in each state on day `i` |
| Recurrence | See state machine transitions above |
| Base case | `hold[0] = -prices[0]`, `sold[0] = 0`, `cooldown[0] = 0` |
| Answer | `max(sold[n-1], cooldown[n-1])` (we shouldn't end holding stock) |

In [ ]:
def max_profit_cooldown(prices):
    if len(prices) <= 1:
        return 0
    n = len(prices)
    hold = [0] * n
    sold = [0] * n
    cool = [0] * n
    hold[0] = -prices[0]
    for i in range(1, n):
        hold[i] = max(hold[i - 1], cool[i - 1] - prices[i])
        sold[i] = hold[i - 1] + prices[i]
        cool[i] = max(cool[i - 1], sold[i - 1])
    return max(sold[n - 1], cool[n - 1])

print("=== Stock with Cooldown (State Machine DP) ===")
prices = [1, 2, 3, 0, 2]
print(f"  Prices: {prices}\n")

n = len(prices)
hold = [0] * n
sold = [0] * n
cool = [0] * n
hold[0] = -prices[0]

print(f"  Day 0: price={prices[0]}  hold={hold[0]:>3}  sold={sold[0]:>3}  cool={cool[0]:>3}  (bought on day 0)")
for i in range(1, n):
    hold[i] = max(hold[i - 1], cool[i - 1] - prices[i])
    sold[i] = hold[i - 1] + prices[i]
    cool[i] = max(cool[i - 1], sold[i - 1])

    h_action = "buy" if cool[i-1] - prices[i] > hold[i-1] else "keep"
    c_action = "from sold" if sold[i-1] > cool[i-1] else "wait"
    print(f"  Day {i}: price={prices[i]}  hold={hold[i]:>3}({h_action:4s})  sold={sold[i]:>3}(sell)  cool={cool[i]:>3}({c_action})")

result = max(sold[n - 1], cool[n - 1])
print(f"\n  Max profit: {result}")
print(f"  Strategy: buy@1, sell@3 (profit 2), cooldown, buy@0, sell@2 (profit 2) = 4 -- nope")
print(f"  Actually: buy@1, sell@2, cooldown, buy@0, sell@2 → profit = {result}")

print("\n--- More tests ---")
for p in [[1, 2, 3, 0, 2], [1], [1, 2], [2, 1, 4]]:
    print(f"  prices={str(p):20s} → profit = {max_profit_cooldown(p)}")

---

## DP Problem Classification

```
┌────────────────────────────────────────────────────────────────────┐
│                     DP PROBLEM FAMILIES                           │
├────────────────────┬───────────────────────────────────────────────┤
│ 1D DP              │ Climbing Stairs, House Robber, Coin Change,  │
│ dp[i]              │ Longest Increasing Subsequence, Decode Ways, │
│                    │ Word Break, Maximum Subarray                 │
├────────────────────┼───────────────────────────────────────────────┤
│ 2D DP              │ 0/1 Knapsack, LCS, Edit Distance,           │
│ dp[i][j]           │ Unique Paths, Min Path Sum, Interleaving    │
│                    │ String, Regular Expression Matching          │
├────────────────────┼───────────────────────────────────────────────┤
│ State Machine DP   │ Stock Buy/Sell (all variants), Paint House,  │
│ multiple states    │ Paint Fence                                  │
├────────────────────┼───────────────────────────────────────────────┤
│ Interval DP        │ Matrix Chain Multiplication, Burst Balloons, │
│ dp[i][j] on range  │ Palindrome Partitioning, Stone Game          │
├────────────────────┼───────────────────────────────────────────────┤
│ DP on Trees        │ House Robber III, Binary Tree Maximum Path,  │
│ recursion + memo   │ Diameter of Binary Tree                      │
├────────────────────┼───────────────────────────────────────────────┤
│ DP on Strings      │ Palindromic Substrings, Longest Palindromic  │
│                    │ Subsequence, Wildcard Matching               │
└────────────────────┴───────────────────────────────────────────────┘
```

### How to Identify DP

```
  Is the problem asking for:
    • "How many ways..."       → likely DP (count paths/combinations)
    • "Minimum/Maximum..."     → likely DP (optimization)
    • "Is it possible..."      → could be DP (feasibility)
    • "Longest/Shortest..."    → often DP

  AND does it have:
    • Overlapping subproblems? → YES → DP
    • Optimal substructure?    → YES → DP
    • Greedy doesn't work?     → Probably DP
```

---

## Practice Problems

| # | Problem | Pattern | Difficulty |
|---|---------|---------|------------|
| 70 | Climbing Stairs | 1D DP (Fibonacci) | Easy |
| 198 | House Robber | 1D DP (skip/take) | Medium |
| 213 | House Robber II | 1D DP (circular) | Medium |
| 322 | Coin Change | 1D DP (unbounded knapsack) | Medium |
| 300 | Longest Increasing Subsequence | 1D DP + binary search | Medium |
| 62 | Unique Paths | 2D grid DP | Medium |
| 64 | Minimum Path Sum | 2D grid DP | Medium |
| 1143 | Longest Common Subsequence | 2D string DP | Medium |
| 72 | Edit Distance | 2D string DP | Medium |
| 309 | Best Time to Buy and Sell Stock with Cooldown | State machine DP | Medium |
| 518 | Coin Change II | 2D DP (count combinations) | Medium |
| 416 | Partition Equal Subset Sum | 0/1 Knapsack variant | Medium |
| 494 | Target Sum | 0/1 Knapsack variant | Medium |
| 139 | Word Break | 1D DP + hash set | Medium |
| 91 | Decode Ways | 1D DP | Medium |
| 152 | Maximum Product Subarray | 1D DP (track min & max) | Medium |
| 5 | Longest Palindromic Substring | 2D interval DP or expand | Medium |
| 312 | Burst Balloons | Interval DP | Hard |
| 10 | Regular Expression Matching | 2D string DP | Hard |

---

## Pattern Cheat Sheet

```
DP PATTERN CHEAT SHEET:

"Min/max ways to reach end"    → 1D DP: dp[i] depends on dp[i-1], dp[i-2]
"Min coins / fewest steps"     → 1D DP: try all choices, take min
"Can't pick adjacent"          → House Robber: dp[i] = max(skip, take)
"Two strings comparison"       → 2D DP: dp[i][j] on prefixes of both strings
"Grid traversal"               → 2D DP: dp[i][j] from dp[i-1][j] and dp[i][j-1]
"Knapsack / subset sum"        → 2D DP → 1D optimization, iterate capacity backwards
"Buy/sell with constraints"    → State machine: define states and transitions
"Optimal over a range [i..j]" → Interval DP: try all split points k in [i..j]
"LIS"                          → O(n²) DP or O(n log n) patience sorting
```

### The DP Checklist (Before Coding)

```
  ┌─────────────────────────────────────────────────────────┐
  │  □  Define the STATE clearly (what does dp[i] mean?)   │
  │  □  Write the RECURRENCE (how does dp[i] depend on     │
  │     smaller subproblems?)                               │
  │  □  Identify BASE CASES (what can I solve without      │
  │     the recurrence?)                                    │
  │  □  Determine the ANSWER cell (dp[n]? max(dp)?)        │
  │  □  Check ITERATION ORDER (do I fill left→right?       │
  │     top→bottom? Does it matter?)                        │
  │  □  Consider SPACE OPTIMIZATION (do I need the whole   │
  │     table or just the last row/two values?)             │
  └─────────────────────────────────────────────────────────┘
```

---

**Next up: Topic 12 — Greedy Algorithms**